# Notebook 01 — Basic RAG (FAISS + citations)

This notebook builds an end-to-end RAG system over local documents:
- load docs from `data/docs/`
- chunk
- embed
- index in FAISS
- retrieve top-k
- generate an answer with **citations**

It supports two modes:
- **API mode**: use an LLM + embedding API if allowed in your environment.
- **Local fallback mode**: local embeddings + a simple grounded response builder.


In [4]:
# Install (Colab)
# If your environment is restricted, you can comment out packages that fail.
!pip -q install langchain langchain-community langchain-text-splitters faiss-cpu numpy sentence-transformers pypdf

In [7]:
import os
from pathlib import Path

# If you store this project on Google Drive, mount it.
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

# Auto-detect project root (works for Drive layouts like MyDrive/RAG/learn RAG)

def detect_project_root() -> Path:
    candidates = [
        Path('/content/learn RAG'),
        Path('/content/drive/MyDrive/learn RAG'),
        Path('/content/drive/MyDrive/RAG/learn RAG'),
        Path('/content/drive/MyDrive/rag/learn RAG'),
    ]
    for c in candidates:
        if (c / 'src' / 'rag_core.py').exists():
            return c

    base = Path('/content/drive/MyDrive')
    if base.exists():
        for sub in base.iterdir():
            if not sub.is_dir():
                continue
            c = sub / 'learn RAG'
            if (c / 'src' / 'rag_core.py').exists():
                return c

    raise FileNotFoundError(
        "Could not find project root. Put the 'learn RAG' folder on Drive (e.g., MyDrive/RAG/learn RAG) or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = detect_project_root()
print('PROJECT_ROOT:', PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Could not find project root. Put the 'learn RAG' folder on Drive (e.g., MyDrive/RAG/learn RAG) or set PROJECT_ROOT manually.

In [5]:
import os
from pathlib import Path

# If you store this project on Google Drive, mount it.
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

# Set PROJECT_ROOT to your folder path.
# Example (Drive): PROJECT_ROOT = Path('/content/drive/MyDrive/learn RAG')
# Example (if uploaded to /content): PROJECT_ROOT = Path('/content/learn RAG')
PROJECT_ROOT = Path('/content/learn RAG')
PROJECT_ROOT

Mounted at /content/drive


PosixPath('/content/learn RAG')

In [8]:
# Add src/ to import path
import sys
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'src'))

from rag_core import build_chunks, load_text_docs

docs = load_text_docs(PROJECT_ROOT / 'data' / 'docs')
print('docs:', len(docs))
print('sample:', docs[0][0] if docs else None)


ModuleNotFoundError: No module named 'rag_core'

In [ ]:
chunks = build_chunks(docs, chunk_chars=1200, overlap_chars=150)
print('chunks:', len(chunks))
print('first chunk metadata:', chunks[0].metadata if chunks else None)


## Embeddings + FAISS

Choose embedding mode:
- `EMBEDDINGS_MODE = 'api'` if you can use an embedding API
- `EMBEDDINGS_MODE = 'local'` if you want to run fully local


In [9]:
EMBEDDINGS_MODE = 'local'  # 'api' or 'local'

texts = [c.text for c in chunks]
metadatas = [c.metadata for c in chunks]

if EMBEDDINGS_MODE == 'local':
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
else:
    # API MODE (choose one): Gemini or OpenAI
    # Gemini example:
    #   os.environ['GOOGLE_API_KEY'] = '...'
    #   from langchain_google_genai import GoogleGenerativeAIEmbeddings
    #   embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
    # OpenAI example:
    #   os.environ['OPENAI_API_KEY'] = '...'
    #   from langchain_openai import OpenAIEmbeddings
    #   embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    raise RuntimeError('Set EMBEDDINGS_MODE and configure an embedding provider.')

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
print('FAISS ready')


NameError: name 'chunks' is not defined

## Ask questions

Choose answering mode:
- `ANSWER_MODE = 'api'`: calls an LLM API (if allowed)
- `ANSWER_MODE = 'grounded_local'`: builds a grounded answer from retrieved chunks without calling an LLM (fallback)


In [ ]:
ANSWER_MODE = 'grounded_local'  # 'api' or 'grounded_local'

def format_citations(docs):
    lines = []
    for d in docs:
        m = d.metadata or {}
        lines.append(f"- {m.get('source_id')}#chunk={m.get('chunk_index')}")
    return "\n".join(lines)

def grounded_local_answer(question: str, docs):
    # Lightweight fallback: provide a short, grounded synthesis + cite sources.
    # This is not a true LLM answer, but it keeps you progressing in restricted environments.
    context = "\n\n".join([d.page_content for d in docs])
    context = context.strip()
    if not context:
        return "I don't know based on the available documents.", ""
    # Simple heuristic: return the most relevant sentences by keyword overlap.
    import re
    q_tokens = set(re.findall(r"[a-zA-Z]{3,}", question.lower()))
    sents = re.split(r"(?<=[.!?])\s+", context)
    scored = []
    for s in sents:
        st = set(re.findall(r"[a-zA-Z]{3,}", s.lower()))
        scored.append((len(q_tokens & st), s))
    scored.sort(key=lambda x: x[0], reverse=True)
    best = [s for score, s in scored[:4] if score > 0]
    if not best:
        best = [s for _, s in scored[:2] if s.strip()]
    answer = " ".join([b.strip() for b in best]).strip()
    return answer, format_citations(docs)

def api_answer(question: str, docs):
    # API MODE (choose one provider)
    # Example Gemini:
    #   !pip install -q langchain-google-genai
    #   os.environ['GOOGLE_API_KEY'] = '...'
    #   from langchain_google_genai import ChatGoogleGenerativeAI
    #   llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash')
    # Example OpenAI:
    #   !pip install -q langchain-openai
    #   os.environ['OPENAI_API_KEY'] = '...'
    #   from langchain_openai import ChatOpenAI
    #   llm = ChatOpenAI(model='gpt-4o-mini')
    raise RuntimeError('Configure your LLM provider in api_answer().')

def ask(question: str):
    docs = retriever.get_relevant_documents(question)
    if ANSWER_MODE == 'grounded_local':
        ans, cites = grounded_local_answer(question, docs)
    else:
        ans, cites = api_answer(question, docs)
    return ans, cites, docs

question = "What are common reliability practices in cloud architecture?"
answer, citations, retrieved = ask(question)
print('Q:', question)
print('\nAnswer:\n', answer)
print('\nCitations:\n', citations)


## What you should do next
- Add 5–20 of your own docs into `data/docs/`
- Change the question(s)
- Verify citations are correct
- If your environment allows, switch `EMBEDDINGS_MODE` and `ANSWER_MODE` to API mode
